# Histograma para $\alpha_{outlier}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def caracterizacion_experimental_outliers_FIXED(muestras_oscuridad, R_medido):
    """
    Calcula el alpha_outlier real buscando el límite físico del ruido del sensor.
    """
    # 1. Limpieza de offsets (Distancia a la mediana de cada píxel)
    medianas = np.median(muestras_oscuridad, axis=0)
    distancias = np.abs(muestras_oscuridad - medianas)
    
    # 2. Normalización por R (unidades de ruido)
    bolsa_desviaciones = distancias.flatten() / R_medido
    
    # 3. CÁLCULO DEL UMBRAL: Usamos un criterio de "Ultra-Confianza"
    # El 99.999% de los eventos de ruido normal están por debajo de este valor.
    # Esto asegura que la línea roja NO corte la montaña.
    alpha_calculado = np.percentile(bolsa_desviaciones, 99.999)
    
    # Agregamos un 10% de margen de seguridad de ingeniería
    alpha_final = alpha_calculado * 1.1

    # --- Gráfico de Verificación ---
    plt.figure(figsize=(12, 6))
    
    # Histograma con más resolución (bins) para ver el final de la falda
    n, bins, patches = plt.hist(bolsa_desviaciones, bins=250, range=(0, 40), log=True, 
                                color='steelblue', edgecolor='black', alpha=0.7, label='Ruido del Sensor')
    
    # Línea del Alpha Calculado
    plt.axvline(x=alpha_final, color='red', linestyle='--', linewidth=3,
                label=f'Umbral de Outlier (Alpha): {alpha_final:.2f}')
    
    # Identificar visualmente los outliers inyectados
    plt.fill_between([alpha_final, 40], 0, 10**6, color='red', alpha=0.1, label='Zona de Detección de Spikes')

    plt.title("CARACTERIZACIÓN FINAL: Límite estadístico del Ruido de Lectura")
    plt.xlabel("Desviación (veces el ruido R)")
    plt.ylabel("Frecuencia (Escala Logarítmica)")
    plt.ylim(0.5, 10**6) # Para ver desde 1 evento hasta un millón
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.legend()
    plt.show()
    
    return alpha_final

# =============================================================================
# TEST CON OUTLIERS REALISTAS
# =============================================================================
R_real = 1.2
data = np.random.normal(loc=15.0, scale=R_real, size=(100, 3648))

# Inyectamos 30 Rayos Cósmicos (Outliers) que DEBEN quedar a la derecha de la línea
for _ in range(30):
    data[np.random.randint(0, 100), np.random.randint(0, 3648)] += np.random.uniform(15, 100)

alpha_v2 = caracterizacion_experimental_outliers_FIXED(data, R_real)

print(f"--- RESULTADO ---")
print(f"Valor experimental de alpha_outlier: {alpha_v2:.2f}")
print(f"Base científica: Percentil 99.999% + 10% de margen de seguridad.")